# finops_bronze_benefits_recommendations

**Purpose**: Lands the shared-scope savings plan recommendation and all alternative commitment levels (coverage / savings / wastage per level) at MCA billing-profile scope, per tenant, for both terms.

**Domain**: finops
**Schema**: bronze

**Inputs**:
- Azure Cost Management `benefitRecommendations` API (api-version 2025-03-01), billing-profile scope, `$expand=properties/allRecommendationDetails`

**Output**: `bronze.benefits_recommendations`

**Parameters** (pipeline via Variable Library):
- `tenants` (string, "" = all configured prefixes)
- `lookback` (string, "Last60Days")

**Trigger**: monthly pipeline, alongside `finops_bronze_benefits_usage`

---

This is the **purchase-decision** dataset: shared-scope recommendations benefit from diversification across subscriptions, so commitment sizing happens here — the per-subscription usage series (`bronze.benefits_usage`) is for visibility and allocation, not sizing. Both P1Y and P3Y are fetched each run so the term trade-off is always current.

One row per (tenant, term, alternative commitment level), with `is_recommended` flagging the engine's pick. Snapshot-append per run date; the snapshot history is what lets you compare `wastage_cost` *predicted* here against *realized* unused commitment in FOCUS (`CommitmentDiscountStatus = 'Unused'`).

**Variable Library additions required**: `{prefix}_billing_scope` per tenant, e.g. `providers/Microsoft.Billing/billingAccounts/{baId}/billingProfiles/{bpId}` (MCA) or `providers/Microsoft.Billing/billingAccounts/{enrollmentId}` (EA). Tenants without it are skipped with a warning.

In [ ]:
%%configure -f
{
    "vCores": 4
}

In [ ]:
%pip install -q azure-identity

In [ ]:
import logging
import time
from datetime import datetime, timezone

import polars as pl
import requests
from azure.identity import ClientSecretCredential
from deltalake import DeltaTable

logger = logging.getLogger(__name__)

In [ ]:
tenants = ""             # "" = all configured tenant prefixes; or "a" / "a,b"
lookback = "Last60Days"  # Last7Days | Last30Days | Last60Days

## 1. Configuration

In [ ]:
ARM = "https://management.azure.com"
API_VERSION = "2025-03-01"
TENANT_PREFIXES = ["a", "b"]
TERMS = ["P1Y", "P3Y"]

VariableLib = notebookutils.variableLibrary.getLibrary("VariableLib")
key_vault_url = VariableLib.key_vault_url
finopshub_root_path = VariableLib.finopshub_root_path
recommendations_table_path = f"{finopshub_root_path}/bronze/benefits_recommendations"

if lookback not in ("Last7Days", "Last30Days", "Last60Days"):
    raise ValueError(f"Invalid lookback: '{lookback}'")

def tenant_config(prefix):
    try:
        cfg = {
            "prefix": prefix,
            "label": f"Tenant {prefix.upper()}",
            "tenant_id": getattr(VariableLib, f"{prefix}_tenant_id"),
            "client_id": getattr(VariableLib, f"{prefix}_client_id"),
            "secret_name": getattr(VariableLib, f"{prefix}_secret_name"),
        }
    except AttributeError:
        return None
    try:
        cfg["billing_scope"] = getattr(VariableLib, f"{prefix}_billing_scope")
    except AttributeError:
        logger.warning("%s configured but %s_billing_scope missing — skipping", cfg["label"], prefix)
        return None
    return cfg

requested = [p.strip() for p in tenants.split(",") if p.strip()] or TENANT_PREFIXES
tenant_configs = [c for p in requested if (c := tenant_config(p)) is not None]
if not tenant_configs:
    raise ValueError(f"No configured tenants with billing scopes match request '{tenants}'")

logger.info("Tenants in scope: %s | lookback=%s terms=%s", [c["label"] for c in tenant_configs], lookback, TERMS)

## 2. ARM Helpers

Same throttle-aware REST pattern as `finops_bronze_benefits_usage` — the shared candidate for `finops-core` once the wheel threshold is met.

In [ ]:
def arm_session(cfg):
    credential = ClientSecretCredential(
        tenant_id=cfg["tenant_id"],
        client_id=cfg["client_id"],
        client_secret=notebookutils.credentials.getSecret(key_vault_url, cfg["secret_name"]),
    )
    token = credential.get_token(f"{ARM}/.default")
    s = requests.Session()
    s.headers["Authorization"] = f"Bearer {token.token}"
    return s

def arm_get(session, url, params=None):
    for _ in range(6):
        r = session.get(url, params=params)
        if r.status_code in (429, 503):
            wait = int(r.headers.get("x-ms-ratelimit-microsoft.consumption-retry-after",
                                     r.headers.get("Retry-After", 10)))
            logger.warning("Throttled (%d), retrying in %ds: %s", r.status_code, wait, url)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Exhausted retries (throttling): {url}")

def arm_get_paged(session, url, params=None):
    items = []
    body = arm_get(session, url, params)
    while True:
        items.extend(body.get("value", []))
        next_link = body.get("nextLink")
        if not next_link:
            return items
        body = arm_get(session, next_link)

## 3. Fetch Recommendations per Tenant and Term

Billing-profile scope returns the Shared-scope recommendation (no per-subscription split — by design). Each alternative in `allRecommendationDetails` becomes a row; `is_recommended` marks the engine's pick by matching `recommendationDetails.commitmentAmount`.

In [ ]:
ingestion_ts = datetime.now(timezone.utc)
snapshot_date = ingestion_ts.date()
rows = []

for cfg in tenant_configs:
    session = arm_session(cfg)
    scope_url = f"{ARM}/{cfg['billing_scope']}/providers/Microsoft.CostManagement/benefitRecommendations"

    for term in TERMS:
        recs = arm_get_paged(session, scope_url, {
            "api-version": API_VERSION,
            "$expand": "properties/allRecommendationDetails",
            "$filter": f"properties/lookBackPeriod eq '{lookback}' AND properties/term eq '{term}'",
        })
        if not recs:
            logger.warning("%s %s: no recommendation returned", cfg["label"], term)
            continue
        if len(recs) > 1:
            logger.warning("%s %s: %d recommendations returned, using first", cfg["label"], term, len(recs))

        props = recs[0]["properties"]
        recommended_amount = (props.get("recommendationDetails") or {}).get("commitmentAmount")
        alternatives = ((props.get("allRecommendationDetails") or {}).get("value")) or []
        if not alternatives:
            logger.warning("%s %s: recommendation has no alternatives list", cfg["label"], term)
            continue

        for alt in alternatives:
            rows.append({
                "tenant_id": cfg["tenant_id"],
                "tenant_label": cfg["label"],
                "billing_scope": cfg["billing_scope"],
                "recommendation_scope": props.get("scope"),
                "term": props.get("term"),
                "look_back_period": props.get("lookBackPeriod"),
                "currency_code": props.get("currencyCode"),
                "arm_sku_name": props.get("armSkuName"),
                "first_consumption_date": props.get("firstConsumptionDate"),
                "last_consumption_date": props.get("lastConsumptionDate"),
                "total_hours": props.get("totalHours"),
                "cost_without_benefit": props.get("costWithoutBenefit"),
                "commitment_amount": alt.get("commitmentAmount"),
                "coverage_percentage": alt.get("coveragePercentage"),
                "average_utilization_percentage": alt.get("averageUtilizationPercentage"),
                "benefit_cost": alt.get("benefitCost"),
                "overage_cost": alt.get("overageCost"),
                "savings_amount": alt.get("savingsAmount"),
                "savings_percentage": alt.get("savingsPercentage"),
                "total_cost": alt.get("totalCost"),
                "wastage_cost": alt.get("wastageCost"),
                "is_recommended": alt.get("commitmentAmount") == recommended_amount,
            })
        logger.info("%s %s: %d alternatives (recommended commitment %s/hr)",
                    cfg["label"], term, len(alternatives), recommended_amount)

logger.info("Total: %d alternative rows", len(rows))

## 4. Write Snapshot — `bronze.benefits_recommendations`

Snapshot-append per run date, same-day rerun replaces (idempotent). The accumulated snapshots are the predicted-wastage history for the forecast-vs-actual variance loop.

In [ ]:
if not rows:
    raise RuntimeError("No recommendations retrieved from any tenant — failing loudly rather than writing an empty snapshot")

df = pl.from_dicts(rows).with_columns(
    pl.lit(ingestion_ts).alias("ingestion_timestamp"),
    pl.lit(f"benefitRecommendations api-version={API_VERSION}").alias("source_file"),
    pl.lit(snapshot_date).alias("snapshot_date"),
)

try:
    DeltaTable(recommendations_table_path)
    table_exists = True
except Exception:
    table_exists = False

if not table_exists:
    df.write_delta(
        recommendations_table_path,
        mode="overwrite",
        delta_write_options={"partition_by": ["snapshot_date"], "engine": "rust"},
    )
else:
    df.write_delta(
        recommendations_table_path,
        mode="overwrite",
        delta_write_options={"predicate": f"snapshot_date = '{snapshot_date}'", "engine": "rust"},
    )

logger.info("Written %d rows to bronze.benefits_recommendations, snapshot_date=%s", len(df), snapshot_date)

## 5. Diagnostics

The commitment lever, per tenant and term — the same table the steering-meeting conversation runs on.

In [ ]:
display(
    df.select([
        "tenant_label", "term", "is_recommended", "commitment_amount",
        "coverage_percentage", "savings_percentage", "wastage_cost",
        "average_utilization_percentage",
    ]).sort(["tenant_label", "term", "commitment_amount"])
)